# Notebook 01 — Exploratory Data Analysis

This notebook explores both datasets before any modelling:
- Class balance
- URL length distributions
- Feature distributions per class
- Correlation heatmap
- Sample URL inspection

**Datasets:**
- D1: UCI Phishing Websites → `data/raw/dataset_phishing_uci.csv`
- D2: PhiUSIIL Phishing URL → `data/raw/phiusiil_phishing_url.csv`

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

matplotlib.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.family': 'DejaVu Sans',
})

from src.data_loader import load_d1, load_d2, generate_synthetic_dataset
from src.features import FEATURE_NAMES

print('Libraries loaded.')

## 1. Load Datasets

If you don't have the real CSVs yet, the cell below falls back to synthetic data.

In [ ]:
D1_PATH = '../data/raw/dataset_phishing_uci.csv'
D2_PATH = '../data/raw/phiusiil_phishing_url.csv'

import os

if os.path.exists(D1_PATH):
    X1, y1 = load_d1(D1_PATH, use_lexical=True)
    print('Loaded real D1 dataset.')
else:
    print('[FALLBACK] D1 CSV not found — using synthetic data.')
    X1, y1 = generate_synthetic_dataset(n_phishing=500, n_legit=500, random_state=42)

if os.path.exists(D2_PATH):
    X2, y2 = load_d2(D2_PATH, sample=10000)  # sample for speed
    print('Loaded real D2 dataset.')
else:
    print('[FALLBACK] D2 CSV not found — using synthetic data.')
    X2, y2 = generate_synthetic_dataset(n_phishing=400, n_legit=600, random_state=99)

print(f'D1 shape: {X1.shape}  |  D2 shape: {X2.shape}')

## 2. Class Balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, y, name in [(axes[0], y1, 'D1 — UCI'), (axes[1], y2, 'D2 — PhiUSIIL')]:
    counts = y.value_counts()
    labels = ['Legitimate', 'Phishing']
    vals   = [counts.get(0, 0), counts.get(1, 0)]
    colors = ['#2ec4b6', '#e63946']
    wedges, texts, autotexts = ax.pie(
        vals, labels=labels, autopct='%1.1f%%',
        colors=colors, startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    )
    for at in autotexts:
        at.set_fontsize(11)
        at.set_fontweight('bold')
    ax.set_title(f'{name}\n(n={len(y):,})', fontweight='bold')

fig.suptitle('Class Balance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/eda_class_balance.png', bbox_inches='tight')
plt.show()
print('Saved: eda_class_balance.png')

## 3. URL Length Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

datasets = [('D1 — UCI', X1, y1), ('D2 — PhiUSIIL', X2, y2)]
length_feats = ['url_length', 'domain_length']

for row_idx, feat in enumerate(length_feats):
    for col_idx, (name, X, y) in enumerate(datasets):
        ax = axes[row_idx][col_idx]
        for cls, label, color in [(0, 'Legitimate', '#2ec4b6'), (1, 'Phishing', '#e63946')]:
            data = X.loc[y == cls, feat].clip(upper=500)
            ax.hist(data, bins=50, alpha=0.6, label=label, color=color, edgecolor='none')
        ax.set_xlabel(feat.replace('_', ' ').title())
        ax.set_ylabel('Count')
        ax.set_title(f'{name} — {feat}', fontweight='bold')
        ax.legend()

fig.suptitle('URL / Domain Length Distributions by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/eda_url_lengths.png', bbox_inches='tight')
plt.show()

## 4. Feature Distributions — Phishing vs Legitimate (D1)

In [ ]:
# Box plots for all 25 features
n_features = len(FEATURE_NAMES)
n_cols = 5
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 3))
axes = axes.flatten()

df1 = X1.copy()
df1['label'] = y1.values

for idx, feat in enumerate(FEATURE_NAMES):
    ax = axes[idx]
    data = [
        df1.loc[df1['label'] == 0, feat].values,
        df1.loc[df1['label'] == 1, feat].values,
    ]
    bp = ax.boxplot(data, patch_artist=True, widths=0.5,
                    medianprops={'color': 'black', 'linewidth': 2})
    bp['boxes'][0].set_facecolor('#2ec4b6')
    bp['boxes'][1].set_facecolor('#e63946')
    ax.set_xticklabels(['Legit', 'Phish'], fontsize=9)
    ax.set_title(feat, fontsize=8, fontweight='bold')
    ax.set_xlabel('')

for j in range(n_features, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature Distributions by Class — D1 (UCI)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../results/figures/eda_feature_boxplots_d1.png', bbox_inches='tight')
plt.show()
print('Saved: eda_feature_boxplots_d1.png')

## 5. Correlation Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, X, name in [(axes[0], X1, 'D1 — UCI'), (axes[1], X2, 'D2 — PhiUSIIL')]:
    corr = X.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(
        corr, mask=mask, ax=ax, cmap='coolwarm', center=0,
        linewidths=0.3, annot=False, fmt='.2f',
        xticklabels=corr.columns, yticklabels=corr.columns,
        vmin=-1, vmax=1,
    )
    ax.set_title(f'{name} — Feature Correlation', fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.savefig('../results/figures/eda_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print('Saved: eda_correlation_heatmap.png')

## 6. Summary Statistics


In [ ]:
print('=== D1 Summary ===')
display(X1.describe().T.round(3))
print()
print('=== D2 Summary ===')
display(X2.describe().T.round(3))

## 7. Observations

- Document your key observations here after viewing the plots.
- Note any class imbalance issues.
- Note which features visually differ most between classes.
- Note differences between D1 and D2 distributions.